In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### create a table saving the info of the new stations want to add

### Need to be careful about the network id in the new run

In [4]:
import pandas as pd

# Manually define station metadata extracted from the paragraph
data = [
    {
        "Native_ID": "452",
        "Station_ID": None,
        "Unique_Name": "Fraser Lake Endako Mines",
        "Network_Name": "BC-Air",
        "lat": 54.034444,
        "lon": -125.095833,
        "elev": 1005
    },
    {
        "Native_ID": "543",
        "Station_ID": None,
        "Unique_Name": "Skookumchuk Farstad Way",
        "Network_Name": "BC-Air",
        "lat": 49.905404,
        "lon": -115.755613,
        "elev": 746
    },
    {
        "Native_ID": "T035",
        "Station_ID": None,
        "Unique_Name": "Horseshoe Bay",
        "Network_Name": "MVRD-AQ",
        "lat": 49.3686,
        "lon": -123.2766,
        "elev": 65
    }
]
# Create DataFrame
stations_df = pd.DataFrame(data)

stations_df

,Native_ID,Station_ID,Unique_Name,Network_Name,lat,lon,elev
0,452,None,Fraser Lake Endako Mines,BC-Air,54.034444,-125.095833,1005
1,543,None,Skookumchuk Farstad Way,BC-Air,49.905404,-115.755613,746
2,T035,None,Horseshoe Bay,MVRD-AQ,49.368600,-123.276600,65


In [6]:
### Get network_id

select_sql = sa.text("""
SELECT network_id
FROM meta_network 
WHERE network_display_name = :network_name
""")

with engine.begin() as conn:
    for idx, row in stations_df.iterrows():

        network_name = row["Network_Name"]

        result = conn.execute(
            select_sql,
            {"network_name": network_name}
        ).fetchall()

        stations_df.at[idx, "network_id"] = ",".join(
              map(str, {r.network_id for r in result})
        )
stations_df

,Native_ID,Station_ID,Unique_Name,Network_Name,lat,lon,elev,network_id
0,452,None,Fraser Lake Endako Mines,BC-Air,54.034444,-125.095833,1005,9
1,543,None,Skookumchuk Farstad Way,BC-Air,49.905404,-115.755613,746,9
2,T035,None,Horseshoe Bay,MVRD-AQ,49.368600,-123.276600,65,72


In [7]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in stations_df.iterrows():
        native_id = row['Native_ID']
        network_id = row['network_id']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(stations_df)}] "
            f"old_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
stations_df['old_station_status'] = exists_list

[  1/3] old_native_id=452             | → NOT EXISTS
[  2/3] old_native_id=543             | → EXISTS
[  3/3] old_native_id=T035            | → NOT EXISTS


In [8]:
from sqlalchemy import func

stations_created = []

# for _, row in df.iloc[0:2].iterrows():
for _, row in stations_df.iterrows():
    
    name = row['Unique_Name']
    nid  = row['Native_ID']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['network_id'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['elev'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Fraser Lake Endako Mines', 14933), ('Skookumchuk Farstad Way', 14934), ('Horseshoe Bay', 14935)]
